In [ ]:
# Install dependencies for phylogenetic analysis 
conda create -n phylo_te -c bioconda -c conda-forge roary iqtree mafft fasttree prokka perl-file-find-rule perl-file-which perl-list-moreutils

conda activate phylo_te

cpanm File::Find::Rule # Install missing Perl module 

conda update --all

conda install -c bioconda blast=2.9.0 # Downgrade blast to avoid compatibility issues with roary

# Verify installation of dependencies 
roary -v
iqtree -v
mafft --version
fasttree -help
prokka --version

### Roary para analise de pangenoma e geracao de alinhamento.

### IQ-TREE para construcao da arvore filogenetica.

### iTOL para visualizacao da arvore e anotacao das IS.

In [ ]:
# 1. anotar genes
prokka --outdir GCF_001_prokka --prefix GCF_001 GCF_001.fasta

## --outdir eh o diretorio de saida, ou seja, onde serao armazenados os arquivos de saida
## --prefix eh o prefixo dos arquivos de saida, ou seja, o nome que sera dado aos arquivos

## NCBI.fasta eh o arquivo de entrada, ou seja, o arquivo que sera anotado, que e o arquivo do genoma do NCBI

### Fazer esta etapa para cada genoma que for ser analisado, ou seja, para cada genoma do NCBI

# MODO Automatizado #################################################################################################################################
## Abra o nano 
nano run_prokka.sh
#---------------------------------------------------------------------------------------------------------------------------------------------------#
# Cria a pasta prokka se ela não existir
mkdir -p prokka

for file in genoma/*.fa genoma/*.fna; do
    # Verifica se o arquivo existe (evita erros se não houver arquivos de um dos tipos)
    [ -e "$file" ] || continue
    
    # Remove a extensão (.fasta ou .fna) do nome base
    base=$(basename "$file" .fa)
    base=$(basename "$base" .fna)
    
    # Remove a parte "_genomic" do nome
    base=${base/_genomic/}
    
    prokka --outdir "prokka/${base}" --prefix "$base" "$file"
done

#---------------------------------------------------------------------------------------------------------------------------------------------------#
### copiar e colar o comando acima 
#### Aperte Ctrl + o para salvar, salvar como run_prokka.sh, aperte Enter para salvar
##### Aperte Ctrl + x para sair do nano

# Dar permissao de execucao ao script
chmod +x run_prokka.sh
# rodar o script 
bash run_prokka.sh


In [ ]:
# criar um diretorio para a coleção de arquivos .gff gerados
cd /mnt/dados/victor_baca/outputs
mkdir -p gff_collection

# copiar os arquivos .gff gerados para o diretorio gff_collection
find /mnt/dados/victor_baca/prokka -type f -name "*.gff" -exec cp {} /mnt/dados/victor_baca/outputs/gff_collection/ \;

In [ ]:
# 2. 
## Na pasta gff_collection, voce tera todos os arquivos .gff gerados pelo prokka
## rodar o roary
nohup roary -e -n -v -f roary_output *.gff &
nohup roary -e -n -v -p 30 -f roary_output *.gff &

### -e: usa alinhamento de nucleotideos 
### -n: nao filtra para apenas um representante por gene 
### -v: modo verboso 
### -p 30: usa 30 threads
### -f: pasta de saida
## coloca o gff do prokka fez
### vai gerar um arquivo .csv e um .aln

In [ ]:
### 3. rodar o iq-tree
cd /mnt/dados/victor_baca/outputs/gff_collection/roary_output
iqtree -s core_gene_alignment.aln -nt AUTO -m GTR+G -bb 1000
### vai gerar um arquivo core_gene_alignment.aln.treefile

In [ ]:
# Antes de ir para o passo 4 vamo transformar os arquivos gerados pelo BLASTn em arquivos .tsv

cd /mnt/dados/victor_baca/outputs
mkdir tsv

## Copiar os arquivos fasta do direotrio cd-hit para a pasta tsv
cp -v /mnt/dados/victor_baca/outputs/cd-hit/blastn/*.fasta /mnt/dados/victor_baca/outputs/tsv/

### Script  Para filtrar arquivos FASTA por TEs específicos, removendo os overlaps ja analisados
nano filter_fasta_tes.py # Copiar o código abaixo

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import re
from pathlib import Path

# Lista de TEs para filtrar
TES = [
    'ISSag9','ISSag3','ISSag5','ISSag2','IS1381A','ISSag8','IS861','ISSag4','ISStin1','ISSag12','ISSpy1','ISCco2','IS1167','ISSag11','ISSmu1','IS1548','IS1216E','IS1167A','ISSg1','IS1550','ISSsu4','ISSth2','ISSa4','IS256','ISLgar1','IS1252','IS6770','IS1194','IS-LL6','IS981','ISLla2','ISSsu5','ISSdy1','ISSdy2','IS200S','ISPsy14','ISPst2','ISPpu17','ISShes10','ISSth1','ISScr1','ISStso1','ISSsu7','ISSsu6','IS1239','ISSpy2','ISSag10','ISLgar5','ISBun2','TnSsu5','Tn5531','Tn551','Tn1546','Tn6246','Tn7114','Tn2670','TnSs1','Tn6674','Tn4001','Tn5393','Tn501','Tn5041','Tn6214','Tn2'
]

def header_contains_te(header, te_list):
    """
    Verifica se o cabeçalho contém algum dos TEs da lista.
    
    Args:
        header: String do cabeçalho FASTA
        te_list: Lista de nomes de TEs
        
    Returns:
        bool: True se contém algum TE, False caso contrário
    """
    header_upper = header.upper()
    for te in te_list:
        # Busca case-insensitive pelo nome do TE
        if te.upper() in header_upper:
            return True
    return False

def filter_fasta(input_file, output_file, te_list):
    """
    Filtra um arquivo FASTA mantendo apenas sequências com TEs específicos.
    
    Args:
        input_file: Caminho do arquivo FASTA de entrada
        output_file: Caminho do arquivo FASTA de saída
        te_list: Lista de nomes de TEs para filtrar
    """
    kept_sequences = 0
    total_sequences = 0
    
    with open(input_file, 'r') as fin, open(output_file, 'w') as fout:
        current_header = None
        current_sequence = []
        keep_sequence = False
        
        for line in fin:
            line = line.rstrip('\n')
            
            if line.startswith('>'):
                # Processar sequência anterior se existir
                if current_header:
                    total_sequences += 1
                    if keep_sequence:
                        fout.write(current_header + '\n')
                        fout.write('\n'.join(current_sequence) + '\n')
                        kept_sequences += 1
                
                # Nova sequência
                current_header = line
                current_sequence = []
                keep_sequence = header_contains_te(line, te_list)
            else:
                # Linha de sequência
                if line.strip():  # Ignorar linhas vazias
                    current_sequence.append(line)
        
        # Processar última sequência
        if current_header:
            total_sequences += 1
            if keep_sequence:
                fout.write(current_header + '\n')
                fout.write('\n'.join(current_sequence) + '\n')
                kept_sequences += 1
    
    return total_sequences, kept_sequences

def main():
    """Função principal para processar todos os arquivos FASTA."""
    
    # Criar diretório de saída
    output_dir = Path('filtered')
    output_dir.mkdir(exist_ok=True)
    
    # Encontrar todos os arquivos .fasta no diretório atual
    fasta_files = list(Path('.').glob('*.fasta'))
    
    if not fasta_files:
        print("❌ Nenhum arquivo .fasta encontrado no diretório atual")
        return
    
    print(f"📂 Encontrados {len(fasta_files)} arquivos FASTA")
    print(f"🔍 Filtrando por {len(TES)} TEs")
    print("=" * 70)
    
    total_kept = 0
    total_processed = 0
    
    for fasta_file in fasta_files:
        # Definir arquivo de saída
        output_file = output_dir / f"{fasta_file.stem}_filter.fasta"
        
        try:
            # Filtrar arquivo
            total_seqs, kept_seqs = filter_fasta(fasta_file, output_file, TES)
            
            percentage = (kept_seqs / total_seqs * 100) if total_seqs > 0 else 0
            
            print(f"✅ {fasta_file.name}")
            print(f"   Total: {total_seqs} | Mantidas: {kept_seqs} ({percentage:.1f}%)")
            print(f"   Saída: {output_file}")
            print()
            
            total_kept += kept_seqs
            total_processed += total_seqs
            
        except Exception as e:
            print(f"❌ Erro ao processar {fasta_file.name}: {e}")
            print()
    
    print("=" * 70)
    print(f"📊 RESUMO FINAL:")
    print(f"   Sequências processadas: {total_processed}")
    print(f"   Sequências mantidas: {total_kept}")
    if total_processed > 0:
        print(f"   Percentual mantido: {total_kept/total_processed*100:.1f}%")
    print(f"   Arquivos gerados em: {output_dir}/")

if __name__ == "__main__":
    main()
#####################################################################################################################################################

#### CTRL + O = Salvar
#### CTRL + X = Sair 

### Tornar o script executável 
chmod +x filter_fasta_tes.py 

python3 filter_fasta_tes.py

### Script para separar os elementos de Transposons (Tns) e Sequências de Inserção (ISs) de arquivos FASTA em diretórios separados
cd /mnt/dados/victor_baca/outputs/tsv/filtered
nano separate_TEs.py

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import re
from pathlib import Path

def create_output_directories():
    """Cria as subpastas Tns e ISs se não existirem"""
    Path("Tns").mkdir(exist_ok=True)
    Path("ISs").mkdir(exist_ok=True)
    print("✓ Diretórios criados: Tns/ e ISs/")

def identify_element_type(header):
    """
    Identifica o tipo de elemento baseado no header.
    Retorna: 'Tn', 'IS', ou None
    """
    # Remove o '>' inicial se houver
    header = header.lstrip('>')
    
    # Padrões mais abrangentes para capturar todos os casos
    # Procura por Tn seguido de letras/números (ex: TnSs1, Tn551, TnSsu5)
    # Procura por In seguido de underscore ou números (ex: In_Tn, In4)
    tn_pattern = r'\b(Tn[A-Za-z0-9]+|In[_0-9])'
    
    # Procura por IS seguido de letras/números (ex: ISSag5, ISCco2, IS1563)
    is_pattern = r'\bIS[A-Za-z0-9]'
    
    # Verifica primeiro por IS (mais específico)
    if re.search(is_pattern, header, re.IGNORECASE):
        return 'IS'
    # Depois verifica por Tn ou In
    elif re.search(tn_pattern, header, re.IGNORECASE):
        return 'Tn'
    
    return None

def process_fasta_file(input_file):
    """
    Processa um arquivo FASTA e separa em Tns e ISs
    """
    base_name = os.path.splitext(os.path.basename(input_file))[0]
    
    # Arquivos de saída
    tn_output = os.path.join("Tns", f"{base_name}_Tns.fasta")
    is_output = os.path.join("ISs", f"{base_name}_ISs.fasta")
    
    # Contadores
    tn_count = 0
    is_count = 0
    other_count = 0
    other_headers = []  # Lista para armazenar headers não classificados
    
    # Abre os arquivos de saída
    with open(tn_output, 'w') as tn_file, \
         open(is_output, 'w') as is_file, \
         open(input_file, 'r') as infile:
        
        current_header = None
        current_sequence = []
        element_type = None
        
        for line in infile:
            line = line.rstrip('\n')
            
            # Nova sequência
            if line.startswith('>'):
                # Salva a sequência anterior se houver
                if current_header and current_sequence:
                    sequence = '\n'.join(current_sequence)
                    
                    if element_type == 'Tn':
                        tn_file.write(f"{current_header}\n{sequence}\n")
                        tn_count += 1
                    elif element_type == 'IS':
                        is_file.write(f"{current_header}\n{sequence}\n")
                        is_count += 1
                    else:
                        other_count += 1
                        other_headers.append(current_header)
                
                # Processa novo header
                current_header = line
                current_sequence = []
                element_type = identify_element_type(line)
                
            else:
                # Adiciona linha de sequência
                if line.strip():  # Ignora linhas vazias
                    current_sequence.append(line)
        
        # Salva a última sequência
        if current_header and current_sequence:
            sequence = '\n'.join(current_sequence)
            
            if element_type == 'Tn':
                tn_file.write(f"{current_header}\n{sequence}\n")
                tn_count += 1
            elif element_type == 'IS':
                is_file.write(f"{current_header}\n{sequence}\n")
                is_count += 1
            else:
                other_count += 1
                other_headers.append(current_header)
    
    return tn_count, is_count, other_count, other_headers

def main():
    """Função principal"""
    print("="*60)
    print("Separador de Elementos Tns e ISs")
    print("="*60)
    
    # Cria diretórios de saída
    create_output_directories()
    
    # Lista todos os arquivos .fasta no diretório atual
    fasta_files = [f for f in os.listdir('.') if f.endswith('.fasta') or f.endswith('.fa')]
    
    if not fasta_files:
        print("\n⚠ Nenhum arquivo FASTA encontrado no diretório atual!")
        return
    
    print(f"\n✓ Encontrados {len(fasta_files)} arquivo(s) FASTA\n")
    
    # Processa cada arquivo
    total_tn = 0
    total_is = 0
    total_other = 0
    all_other_headers = []  # Lista para todos os headers não classificados
    
    for fasta_file in sorted(fasta_files):
        print(f"Processando: {fasta_file}")
        tn_count, is_count, other_count, other_headers = process_fasta_file(fasta_file)
        
        print(f"  ├─ Transposons (Tns): {tn_count}")
        print(f"  ├─ Seq. Inserção (ISs): {is_count}")
        print(f"  └─ Outros elementos: {other_count}")
        
        # Se houver outros elementos, mostra os headers
        if other_count > 0:
            print(f"\n  ⚠ Headers não classificados em {fasta_file}:")
            for header in other_headers:
                print(f"     {header}")
            all_other_headers.extend([(fasta_file, header) for header in other_headers])
        
        print()
        
        total_tn += tn_count
        total_is += is_count
        total_other += other_count
    
    # Resumo final
    print("="*60)
    print("RESUMO GERAL")
    print("="*60)
    print(f"Total de Transposons (Tns): {total_tn}")
    print(f"Total de Seq. Inserção (ISs): {total_is}")
    print(f"Total de outros elementos: {total_other}")
    
    # Mostra resumo dos elementos não classificados
    if total_other > 0:
        print("\n" + "="*60)
        print("ELEMENTOS NÃO CLASSIFICADOS - RESUMO")
        print("="*60)
        for fasta_file, header in all_other_headers:
            print(f"[{fasta_file}] {header}")
    
    print("\n✓ Processamento concluído!")
    print(f"  ├─ Arquivos Tns salvos em: ./Tns/")
    print(f"  └─ Arquivos ISs salvos em: ./ISs/")

if __name__ == "__main__":
    main()
#####################################################################################################################################################

#### CTRL + O = Salvar
#### CTRL + X = Sair 

chmod +x separate_TEs.py
python3 separate_TEs.py

### Como o elemento ISPa12_ISPpu17 ainda não foi filtrado por ter o ISPpu17 (ISPpu17 um elemento desejado, mais não o ISPa12_ISPpu17), vou criar um novo script para remover/filtrar esse elemento ISPa12_ISPpu17 dos meus arquivos fastas dos elementos ISs

cd /mnt/dados/victor_baca/outputs/tsv/filtered/ISs # Diretório onde estão os arquivos FASTA dos elementos ISs do BLASTn

nano filter_isppa12.py # Copiar o código abaixo

#####################################################################################################################################################
#!/usr/bin/env python3
"""
Script para filtrar sequências FASTA que contêm ISPa12_ISPpu17 no campo Name.
Remove essas sequências de todos os arquivos FASTA no diretório especificado
e salva os resultados em uma nova subpasta.
"""

import os
import re
from pathlib import Path

def parse_fasta(file_path):
    """
    Lê um arquivo FASTA e retorna uma lista de tuplas (header, sequence).
    """
    sequences = []
    current_header = None
    current_seq = []
    
    with open(file_path, 'r') as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('>'):
                # Salva a sequência anterior se existir
                if current_header is not None:
                    sequences.append((current_header, ''.join(current_seq)))
                # Inicia nova sequência
                current_header = line
                current_seq = []
            else:
                current_seq.append(line)
        
        # Salva a última sequência
        if current_header is not None:
            sequences.append((current_header, ''.join(current_seq)))
    
    return sequences

def has_isppa12_isppu17(header):
    """
    Verifica se o header contém ISPa12_ISPpu17 no campo Name.
    """
    # Procura por Name=...ISPa12_ISPpu17...
    match = re.search(r'Name=([^;]+)', header)
    if match:
        name_value = match.group(1)
        if 'ISPa12_ISPpu17' in name_value:
            return True
    return False

def filter_fasta(input_file, output_file):
    """
    Filtra um arquivo FASTA removendo sequências com ISPa12_ISPpu17 no Name.
    Retorna o número de sequências removidas e mantidas.
    """
    sequences = parse_fasta(input_file)
    
    filtered_sequences = []
    removed_count = 0
    
    for header, seq in sequences:
        if has_isppa12_isppu17(header):
            removed_count += 1
        else:
            filtered_sequences.append((header, seq))
    
    # Escreve o arquivo filtrado
    with open(output_file, 'w') as f:
        for header, seq in filtered_sequences:
            f.write(f"{header}\n{seq}\n")
    
    return removed_count, len(filtered_sequences)

def main():
    # Diretório de entrada
    input_dir = Path('/mnt/dados/victor_baca/outputs/tsv/filtered/ISs')
    
    # Cria o diretório de saída
    output_dir = input_dir / 'filtered_no_ISPa12_ISPpu17'
    output_dir.mkdir(exist_ok=True)
    
    print(f"Diretório de entrada: {input_dir}")
    print(f"Diretório de saída: {output_dir}")
    print("-" * 70)
    
    # Processa todos os arquivos .fasta no diretório
    fasta_files = list(input_dir.glob('*.fasta'))
    
    if not fasta_files:
        print("Nenhum arquivo FASTA encontrado no diretório!")
        return
    
    total_removed = 0
    total_kept = 0
    
    for fasta_file in fasta_files:
        print(f"\nProcessando: {fasta_file.name}")
        
        output_file = output_dir / fasta_file.name
        
        try:
            removed, kept = filter_fasta(fasta_file, output_file)
            total_removed += removed
            total_kept += kept
            
            print(f"  ✓ Sequências removidas: {removed}")
            print(f"  ✓ Sequências mantidas: {kept}")
            print(f"  ✓ Arquivo salvo: {output_file.name}")
            
        except Exception as e:
            print(f"  ✗ Erro ao processar {fasta_file.name}: {e}")
    
    print("\n" + "=" * 70)
    print(f"RESUMO:")
    print(f"  Arquivos processados: {len(fasta_files)}")
    print(f"  Total de sequências removidas: {total_removed}")
    print(f"  Total de sequências mantidas: {total_kept}")
    print(f"  Arquivos salvos em: {output_dir}")
    print("=" * 70)

if __name__ == "__main__":
    main()
#####################################################################################################################################################

#### CTRL + O -> para salvar o arquivo
#### CTRL + X -> para sair do editor

chmod +x filter_isppa12.py

python3 filter_isppa12.py

###
cd /mnt/dados/victor_baca/outputs/tsv
mkdir tsv

In [ ]:
# 4. criar um novo arquivo .tsv com os arquivos .tsv do ISEScan
''' genome  IS3     IS256   IS30
    GCF_001 2       0       1
    GCF_002 1       1       0
    GCF_003 3       2       1'''

# Script para converter arquivos FASTA (blast hits) em formato TSV
nano fasta_to_tsv_converter.py

#####################################################################################################################################################
#!/usr/bin/env python3
"""
Script para converter arquivos FASTA (blast hits) em formato TSV
Processa arquivos dos diretórios Tns e filtered_no_ISPa12_ISPpu17
"""

import os
import re
from pathlib import Path
from typing import Dict, List


def parse_fasta_header(header: str) -> Dict[str, str]:
    """
    Extrai informações do cabeçalho FASTA
    
    Exemplo de header:
    >blast_hit_25 NODE_11_length_58457_cov_32.323058:1-129(-) ISFinder_IS1381A 
    ID=blast_hit_25;Name=ISFinder_IS1381A;Target=ISFinder_IS1381A 159 31;
    identity=99.22;query_coverage=0.00;evalue=2.83e-59;bitscore=233.0;
    alignment_length=129;mismatches=1;gap_openings=0;query_length=58457;
    subject_length=900;subject_frame=-1
    """
    data = {}
    
    # Remove o '>' inicial
    header = header.lstrip('>')
    
    # Divide em partes principais
    parts = header.split(maxsplit=2)
    
    if len(parts) >= 1:
        data['blast_hit_id'] = parts[0]
    
    if len(parts) >= 2:
        # Parse coordenadas (ex: NODE_11_length_58457_cov_32.323058:1-129(-))
        coord_match = re.search(r'(.+):(\d+)-(\d+)\(([+-])\)', parts[1])
        if coord_match:
            data['sequence_id'] = coord_match.group(1)
            data['start'] = coord_match.group(2)
            data['end'] = coord_match.group(3)
            data['strand'] = coord_match.group(4)
    
    if len(parts) >= 3:
        # Nome do IS/Tn (terceira parte antes dos parâmetros)
        name_part = parts[2].split()[0] if parts[2] else ''
        data['is_tn_name'] = name_part
    
    # Parse dos parâmetros (ID=..;Name=..;...)
    params_match = re.search(r'ID=([^;]+);.*', header)
    if params_match:
        params_str = header[header.find('ID='):]
        
        # Extrai cada parâmetro
        param_patterns = {
            'ID': r'ID=([^;]+)',
            'Name': r'Name=([^;]+)',
            'Target': r'Target=([^;]+)',
            'identity': r'identity=([^;]+)',
            'query_coverage': r'query_coverage=([^;]+)',
            'evalue': r'evalue=([^;]+)',
            'bitscore': r'bitscore=([^;]+)',
            'alignment_length': r'alignment_length=([^;]+)',
            'mismatches': r'mismatches=([^;]+)',
            'gap_openings': r'gap_openings=([^;]+)',
            'query_length': r'query_length=([^;]+)',
            'subject_length': r'subject_length=([^;]+)',
            'subject_frame': r'subject_frame=([^;\n]+)'
        }
        
        for param_name, pattern in param_patterns.items():
            match = re.search(pattern, params_str)
            if match:
                data[param_name] = match.group(1).strip()
    
    return data


def fasta_to_tsv(fasta_file: str, tsv_file: str) -> int:
    """
    Converte um arquivo FASTA em TSV
    
    Args:
        fasta_file: Caminho do arquivo FASTA de entrada
        tsv_file: Caminho do arquivo TSV de saída
    
    Returns:
        Número de sequências processadas
    """
    sequences = []
    current_header = None
    current_sequence = []
    
    # Lê o arquivo FASTA
    with open(fasta_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith('>'):
                # Salva a sequência anterior se existir
                if current_header is not None:
                    seq_data = parse_fasta_header(current_header)
                    seq_data['sequence'] = ''.join(current_sequence)
                    seq_data['sequence_length'] = len(seq_data['sequence'])
                    sequences.append(seq_data)
                
                current_header = line
                current_sequence = []
            else:
                current_sequence.append(line)
        
        # Salva a última sequência
        if current_header is not None:
            seq_data = parse_fasta_header(current_header)
            seq_data['sequence'] = ''.join(current_sequence)
            seq_data['sequence_length'] = len(seq_data['sequence'])
            sequences.append(seq_data)
    
    # Escreve o arquivo TSV
    if sequences:
        # Define as colunas na ordem desejada
        columns = [
            'blast_hit_id', 'sequence_id', 'start', 'end', 'strand',
            'is_tn_name', 'ID', 'Name', 'Target', 'identity',
            'query_coverage', 'evalue', 'bitscore', 'alignment_length',
            'mismatches', 'gap_openings', 'query_length', 'subject_length',
            'subject_frame', 'sequence_length', 'sequence'
        ]
        
        with open(tsv_file, 'w') as f:
            # Escreve cabeçalho
            f.write('\t'.join(columns) + '\n')
            
            # Escreve dados
            for seq in sequences:
                row = [seq.get(col, '') for col in columns]
                f.write('\t'.join(str(val) for val in row) + '\n')
    
    return len(sequences)


def process_directory(input_dir: str, output_dir: str) -> Dict[str, int]:
    """
    Processa todos os arquivos .fasta em um diretório
    
    Args:
        input_dir: Diretório de entrada com arquivos .fasta
        output_dir: Diretório de saída para arquivos .tsv
    
    Returns:
        Dicionário com estatísticas do processamento
    """
    # Cria diretório de saída se não existir
    os.makedirs(output_dir, exist_ok=True)
    
    stats = {
        'total_files': 0,
        'processed_files': 0,
        'total_sequences': 0,
        'errors': []
    }
    
    # Lista todos os arquivos .fasta
    input_path = Path(input_dir)
    fasta_files = list(input_path.glob('*.fasta'))
    
    stats['total_files'] = len(fasta_files)
    
    for fasta_file in sorted(fasta_files):
        try:
            # Define nome do arquivo de saída
            tsv_filename = fasta_file.stem + '.tsv'
            tsv_file = os.path.join(output_dir, tsv_filename)
            
            # Converte
            num_sequences = fasta_to_tsv(str(fasta_file), tsv_file)
            
            stats['processed_files'] += 1
            stats['total_sequences'] += num_sequences
            
            print(f"✓ Processado: {fasta_file.name} -> {tsv_filename} ({num_sequences} sequências)")
            
        except Exception as e:
            error_msg = f"Erro ao processar {fasta_file.name}: {str(e)}"
            stats['errors'].append(error_msg)
            print(f"✗ {error_msg}")
    
    return stats


def main():
    """Função principal"""
    # Define os diretórios base
    base_dir = "/mnt/dados/victor_baca/outputs/tsv/filtered"
    
    # Diretórios de entrada e saída
    directories = [
        {
            'input': os.path.join(base_dir, "ISs", "filtered_no_ISPa12_ISPpu17"),
            'output': os.path.join(base_dir, "ISs", "filtered_no_ISPa12_ISPpu17_tsv")
        },
        {
            'input': os.path.join(base_dir, "Tns"),
            'output': os.path.join(base_dir, "Tns_tsv")
        }
    ]
    
    print("=" * 80)
    print("Conversor FASTA para TSV - Blast Hits")
    print("=" * 80)
    print()
    
    total_stats = {
        'total_files': 0,
        'processed_files': 0,
        'total_sequences': 0,
        'errors': []
    }
    
    # Processa cada diretório
    for dir_config in directories:
        input_dir = dir_config['input']
        output_dir = dir_config['output']
        
        print(f"\nProcessando diretório: {input_dir}")
        print(f"Saída em: {output_dir}")
        print("-" * 80)
        
        if not os.path.exists(input_dir):
            print(f"⚠ AVISO: Diretório não encontrado: {input_dir}")
            continue
        
        stats = process_directory(input_dir, output_dir)
        
        # Acumula estatísticas
        total_stats['total_files'] += stats['total_files']
        total_stats['processed_files'] += stats['processed_files']
        total_stats['total_sequences'] += stats['total_sequences']
        total_stats['errors'].extend(stats['errors'])
        
        print()
        print(f"Arquivos processados: {stats['processed_files']}/{stats['total_files']}")
        print(f"Total de sequências: {stats['total_sequences']}")
    
    # Resumo final
    print("\n" + "=" * 80)
    print("RESUMO FINAL")
    print("=" * 80)
    print(f"Total de arquivos processados: {total_stats['processed_files']}/{total_stats['total_files']}")
    print(f"Total de sequências convertidas: {total_stats['total_sequences']}")
    
    if total_stats['errors']:
        print(f"\n⚠ Erros encontrados: {len(total_stats['errors'])}")
        for error in total_stats['errors']:
            print(f"  - {error}")
    else:
        print("\n✓ Todos os arquivos processados com sucesso!")
    
    print()


if __name__ == "__main__":
    main()
#####################################################################################################################################################

chmod +x fasta_to_tsv_converter.py
python3 fasta_to_tsv_converter.py

# Codigo para fazer a copia de todos os arquivos .tsv de um determinado diretorio para outro (sem importar se esta em subdiretorios)
cd /mnt/dados/victor_baca/outputs/tsv
mkdir ISEScan
find '/mnt/dados/victor_baca/outputs/isescan' -type f -name "*.tsv" -exec cp -t '/mnt/dados/victor_baca/outputs/tsv/ISEScan' {} +

# Agora vou criar um script para renomear os arquivos .tsv com os nomes das cepas ao invés dos IDs do NCBI, isso facilitara a edição depois
nano rename_tsv_files.py

#####################################################################################################################################################
#!/usr/bin/env python3

import os
import shutil
from pathlib import Path

# Mapeamento NCBI ID -> Strain name baseado no arquivo "NCBI - Complete Genoma.txt"
ncbi_to_strain = {
    'GCF_002812445.1': 'SGEHI2015-95',
    'GCF_002812465.1': 'SGEHI2015-25',
    'GCF_002812505.1': 'SGEHI2015-113',
    'GCF_002812425.1': 'SGEHI2015-107',
    'GCF_000636115.1': '138spar',
    'GCF_001190805.1': 'HN016',
    'GCF_003939065.1': 'TFJ0901',
    'GCF_041728555.1': 'SA2BKE',
    'GCF_013000945.1': '01173',
    'GCA_003186145.1': 'QMA0271',
    'GCF_000967445.1': '2-22',
    'GCA_002881395.1': 'SA102',
    'GCF_002881775.1': 'SA627',
    'GCF_002882835.1': 'SA623',
    'GCF_002882745.1': 'SA375',
    'GCF_002881695.1': 'SA374',
    'GCF_002882645.1': 'SA346',
    'GCF_002881615.1': 'SA343',
    'GCF_002882555.1': 'SA341',
    'GCF_002882465.1': 'SA333',
    'GCF_002882375.1': 'SA330',
    'GCF_002881595.1': 'SA289',
    'GCF_002881575.1': 'SA256',
    'GCF_002881555.1': 'SA245',
    'GCF_002881535.1': 'SA220',
    'GCF_002881515.1': 'SA218',
    'GCF_002882275.1': 'SA212',
    'GCF_002881495.1': 'SA209',
    'GCF_002882205.1': 'SA201',
    'GCF_002882125.1': 'SA191',
    'GCF_002881475.1': 'SA195',
    'GCF_002881455.1': 'SA184',
    'GCA_002881435.1': 'SA159',
    'GCF_002881415.1': 'SA136',
    'GCF_002882035.1': 'SA132',
    'GCF_002881935.1': 'SA97',
    'GCF_002881375.1': 'SA95',
    'GCF_002881355.1': 'SA85',
    'GCF_002881335.1': 'SA81',
    'GCF_002881315.1': 'SA79',
    'GCF_002881865.1': 'SA75',
    'GCF_002881295.1': 'SA16',
    'GCF_002881255.1': 'SA05',
    'GCF_002881215.1': 'SA01',
    'GCF_002881195.1': 'SA73',
    'GCF_002881235.1': 'SA53',
    'GCF_002881175.1': 'SA33',
    'GCF_002881155.1': 'SA30',
    'GCF_000302475.2': 'SA020',
    'GCF_002197205.1': 'CUGBS591',
    'GCF_001190885.1': 'H002',
    'GCF_001592425.1': 'CU_GBS_98',
    'GCF_001592385.1': 'CU_GBS_08',
    'GCF_006716245.1': '32790-3A',
    'GCF_001266635.1': 'GBS85147',
    'GCF_001552035.1': 'NGBS128',
    'GCF_001026925.1': 'SS1',
    'GCF_000831125.1': 'GBS2-NM',
    'GCF_000831145.1': 'GBS1-NY',
    'GCF_000831105.1': 'GBS6',
    'GCF_025402775.1': 'TAH10_84dMrvR',
    'GCF_025402755.1': 'TAH10_84dUdp',
    'GCF_025402735.1': 'TAH10_84',
    'GCF_021496865.1': 'SS43',
    'GCF_021496845.1': 'SS44',
    'GCF_021496825.1': 'SS102',
    'GCF_021496945.1': 'SS111',
    'GCF_021496805.1': 'SS114',
    'GCF_019552345.1': 'SS1168',
    'GCF_030078235.1': 'CNCTC10_84dCas9',
    'GCF_030078295.1': 'CNCTC10_84sCas9',
    'GCF_030078315.1': 'CNCTC10_84sCas9revertant',
    'GCF_030078355.1': 'CNCTC10_84Cas9AEKOrevertant',
    'GCF_030078375.1': 'CNCTC10_84Cas9AEKO',
    'GCF_030078255.1': 'A909_sCas9',
    'GCF_030078275.1': 'A909_sCas9revertant',
    'GCF_030078335.1': 'A909_dCas9',
    'GCF_030078395.1': 'A909_Cas9_AEKOrevertant',
    'GCF_030078415.1': 'A909_Cas9_AEKO',
    'GCF_003966545.1': 'HU-GS5823',
    'GCF_030323365.1': 'GBSIR0001',
    'GCF_021655535.1': 'GCMC97051',
    'GCF_002289205.1': '874391',
    'GCF_001275545.2': 'SG-M1',
    'GCF_002197385.1': 'SG-M4',
    'GCF_002197365.1': 'SG-M6',
    'GCF_002197325.1': 'SG-M8',
    'GCF_002197305.1': 'SG-M25',
    'GCF_002197285.1': 'SG-M29',
    'GCF_002197245.1': 'SG-M50',
    'GCF_002197265.1': 'SG-M158',
    'GCF_002197425.1': 'SG-M163',
    'GCA_016454805.1': '2012-845',
    'GCF_030489725.1': 'NEM316',
    'GCF_002214425.1': 'C001',
    'GCF_000427035.1': '09mas018883',
    'GCF_900078265.1': 'SA111',
    'GCF_947313835.2': 'MRI_Z2-336',
    'GCF_029625275.1': 'S1',
    'GCF_009930895.1': 'YZ1605',
    'GCF_009930915.1': 'NJ1606',
    'GCF_029625255.1': 'S5',
    'GCF_001708205.1': 'M19',
    'GCF_025631115.1': 'BAU_MH_Bag_2013',
    'GCF_025631095.1': 'BAU_MH_Bag_2010',
    'GCF_025631125.1': 'BAU_MH_Bag_2014',
    'GCF_025801965.1': 'BR-MHR519',
    'GCF_027533695.1': 'BR-MHR951',
    'GCF_024424535.1': 'LGMAI_St_14',
    'GCF_024424555.1': 'LGMAI_St_08',
    'GCF_024424515.1': 'LGMAI_St_11',
    'GCF_014895495.1': 'TA_B490',
    'GCA_020980705.1': 'GBS181',
    'GCA_020980735.1': 'GBS180',
    'GCA_020980665.1': 'GBS183',
    'GCF_020980675.1': 'GBS182',
    'GCA_020980325.1': 'GBS185',
    'GCA_020980385.1': 'GBS179',
    'GCF_020980305.1': 'GBS184',
    'GCF_020980285.1': 'GBS186',
    'GCF_019794075.1': 'HFJ14',
    'GCF_019794115.1': 'PZD1',
    'GCF_019794045.1': 'PZD2',
    'GCF_019794085.1': 'PZD12',
    'GCF_019794035.1': 'PZD29',
    'GCF_019794015.1': 'PZD30',
    'GCF_019793995.1': 'G1-9',
    'GCF_019793945.1': 'G6-23',
    'GCF_019794635.1': 'G9',
    'GCF_019794875.1': 'HFJ96',
    'GCF_006543535.1': '1B13M',
    'GCF_006543565.1': '3B11V',
    'GCF_006543545.1': '3B13R',
    'GCF_006543525.1': '3B21M',
    'GCF_006543465.1': '4B14M',
    'GCF_006543275.1': '5B2M',
    'GCF_006543375.1': '5B15M',
    'GCF_006543335.1': '5B20M',
    'GCF_006543305.1': '5B21V',
    'GCF_006543265.1': '5B28V',
    'GCF_006543115.1': '6B19M',
    'GCF_006543125.1': '7B18M',
    'GCF_006543135.1': '8B14M',
    'GCF_003605605.1': '3966RFQB',
    'GCA_020981825.1': 'GBS110',
    'GCA_020981865.1': 'GBS111',
    'GCA_020981795.1': 'GBS112',
    'GCA_020981845.1': 'GBS113',
    'GCA_020981705.1': 'GBS114',
    'GCA_020981685.1': 'GBS115',
    'GCA_020980445.1': 'GBS109',
    'GCA_020980475.1': 'GBS116',
    'GCA_020980425.1': 'GBS117',
    'GCF_020980405.1': 'GBS160',
    'GCA_020980155.1': 'GBS95',
}

# Adicionar também as cepas SA que aparecem no comando ls
strain_files = {
    'SA10-UEL': 'SA10-UEL',
    'SA11-UEL': 'SA11-UEL',
    'SA9-UEL': 'SA9-UEL',
    'SA8-UEL': 'SA8-UEL',
}


def rename_files_in_directory(directory_path, dry_run=True):
    """
    Renomeia arquivos .tsv em um diretório baseado no mapeamento NCBI -> Strain
    
    Args:
        directory_path: Caminho do diretório contendo os arquivos .tsv
        dry_run: Se True, apenas mostra o que seria feito sem executar
    
    Returns:
        dict: Dicionário com estatísticas da operação
    """
    stats = {
        'renamed': 0,
        'not_found': 0,
        'already_exists': 0,
        'errors': 0
    }
    
    if not os.path.exists(directory_path):
        print(f"❌ Diretório não encontrado: {directory_path}")
        return stats
    
    print(f"\n{'=' * 80}")
    print(f"Processando diretório: {directory_path}")
    print(f"Modo: {'DRY RUN (simulação)' if dry_run else 'EXECUÇÃO REAL'}")
    print(f"{'=' * 80}\n")
    
    # Listar todos os arquivos .tsv no diretório
    tsv_files = sorted([f for f in os.listdir(directory_path) if f.endswith('.tsv')])
    
    if not tsv_files:
        print("⚠️  Nenhum arquivo .tsv encontrado no diretório")
        return stats
    
    print(f"📁 Encontrados {len(tsv_files)} arquivos .tsv\n")
    
    for filename in tsv_files:
        old_path = os.path.join(directory_path, filename)
        
        # Verificar se já é um arquivo com nome de cepa
        if filename in strain_files.values() or any(filename.startswith(strain) for strain in strain_files.values()):
            print(f"⏭️  IGNORADO: {filename} (já está com nome de cepa)")
            continue
        
        # Extrair o NCBI ID do nome do arquivo
        ncbi_id = None
        for key in ncbi_to_strain.keys():
            if filename.startswith(key):
                ncbi_id = key
                break
        
        if not ncbi_id:
            print(f"⚠️  NÃO MAPEADO: {filename}")
            stats['not_found'] += 1
            continue
        
        # Obter o nome da cepa
        strain_name = ncbi_to_strain[ncbi_id]
        
        # Determinar o sufixo do arquivo (o que vem depois do NCBI ID)
        # Exemplos: _blastn_filter_ISs.tsv, _blastn_filter_Tns.tsv
        suffix = filename.replace(ncbi_id, '').split('_genomic', 1)[-1] if '_genomic' in filename else '.tsv'
        if not suffix.startswith('_'):
            suffix = '.tsv'
        else:
            # Simplificar o sufixo mantendo apenas a parte relevante
            if '_blastn_filter_ISs.tsv' in suffix:
                suffix = '_ISs.tsv'
            elif '_blastn_filter_Tns.tsv' in suffix:
                suffix = '_Tns.tsv'
            else:
                suffix = '.tsv'
        
        new_filename = f"{strain_name}{suffix}"
        new_path = os.path.join(directory_path, new_filename)
        
        # Verificar se o novo arquivo já existe
        if os.path.exists(new_path) and new_path != old_path:
            print(f"⚠️  JÁ EXISTE: {filename} -> {new_filename}")
            stats['already_exists'] += 1
            continue
        
        # Executar ou simular a renomeação
        try:
            if dry_run:
                print(f"✅ SERIA RENOMEADO: {filename} -> {new_filename}")
            else:
                os.rename(old_path, new_path)
                print(f"✅ RENOMEADO: {filename} -> {new_filename}")
            stats['renamed'] += 1
        except Exception as e:
            print(f"❌ ERRO ao renomear {filename}: {str(e)}")
            stats['errors'] += 1
    
    return stats


def process_all_directories(base_paths, dry_run=True):
    """
    Processa todos os diretórios especificados
    
    Args:
        base_paths: Lista de caminhos dos diretórios a processar
        dry_run: Se True, apenas mostra o que seria feito sem executar
    """
    total_stats = {
        'renamed': 0,
        'not_found': 0,
        'already_exists': 0,
        'errors': 0
    }
    
    for path in base_paths:
        stats = rename_files_in_directory(path, dry_run)
        for key in total_stats:
            total_stats[key] += stats[key]
    
    print(f"\n{'=' * 80}")
    print("RESUMO GERAL")
    print(f"{'=' * 80}")
    print(f"✅ Arquivos renomeados: {total_stats['renamed']}")
    print(f"⚠️  Não mapeados: {total_stats['not_found']}")
    print(f"⚠️  Já existem: {total_stats['already_exists']}")
    print(f"❌ Erros: {total_stats['errors']}")
    print(f"{'=' * 80}\n")


if __name__ == "__main__":
    # IMPORTANTE: Ajuste estes caminhos de acordo com seu sistema
    directories_to_process = [
        "/mnt/dados/victor_baca/outputs/tsv/filtered/ISs/filtered_no_ISPa12_ISPpu17_tsv",
        "/mnt/dados/victor_baca/outputs/tsv/filtered/Tns_tsv/",
        "/mnt/dados/victor_baca/outputs/tsv/ISEScan/",
    ]
    
    print("""
╔════════════════════════════════════════════════════════════════════════════╗
║          SCRIPT DE RENOMEAÇÃO DE ARQUIVOS TSV - GENOMAS NCBI              ║
╚════════════════════════════════════════════════════════════════════════════╝
    """)
    
    # Primeiro executar em modo DRY RUN para verificar
    print("🔍 FASE 1: Simulação (Dry Run) - Nenhum arquivo será modificado\n")
    process_all_directories(directories_to_process, dry_run=True)
    
    # Solicitar confirmação do usuário
    print("\n⚠️  ATENÇÃO: Deseja executar a renomeação REAL dos arquivos?")
    print("Digite 'SIM' (em maiúsculas) para confirmar ou qualquer outra coisa para cancelar:")
    user_input = input().strip()
    
    if user_input == "SIM":
        print("\n🚀 FASE 2: Execução Real - Renomeando arquivos...\n")
        process_all_directories(directories_to_process, dry_run=False)
        print("✅ Processo concluído com sucesso!")
    else:
        print("\n❌ Operação cancelada pelo usuário. Nenhum arquivo foi modificado.")
#####################################################################################################################################################

chmod +x rename_tsv_files.py
python3 rename_tsv_files.py

# Movendo todos os arquivos .tsv para uma unica pasta, a pasta tsv
mv /mnt/dados/victor_baca/outputs/tsv/ISEScan /mnt/dados/victor_baca/outputs/tsv/tsv 
mv /mnt/dados/victor_baca/outputs/tsv/filtered/ISs/filtered_no_ISPa12_ISPpu17_tsv /mnt/dados/victor_baca/outputs/tsv/tsv
mv /mnt/dados/victor_baca/outputs/tsv/filtered/Tns_tsv/ /mnt/dados/victor_baca/outputs/tsv/tsv
cd /mnt/dados/victor_baca/outputs/tsv/tsv

# Criar Script para gerar o arquivo is_counts.tsv a partir dos arquivos .tsv em múltiplos subdiretórios
## Esse arquivo is_counts.tsv deve conter a contagem de todas as familias de TEs (ISEScan) e elementos IS/Tn (Tns_tsv e filtered_no_ISPa12_ISPpu17_tsv) por genoma
nano generate_is_counts.py
#####################################################################################################################################################
#!/usr/bin/env python3
"""
Script para gerar is_counts.tsv a partir de arquivos TSV em múltiplos subdiretórios.

Processa arquivos de três fontes:
- ISEScan/: usa coluna 'family' (famílias de TEs)
- Tns_tsv/: usa coluna 'is_tn_name' (elementos IS/Tn)
- filtered_no_ISPa12_ISPpu17_tsv/: usa coluna 'is_tn_name' (elementos IS/Tn)

IMPORTANTE: 
1. Consolida dados do mesmo genoma de diferentes arquivos
2. Normaliza nomes de elementos IS/Tn para consolidar variantes
3. Remove elementos inválidos ou vazios
4. Gera UM único arquivo is_counts.tsv com todas as famílias e elementos
"""

import csv
import os
import re
from collections import defaultdict
from pathlib import Path

# Diretório base contendo os subdiretórios
BASE_DIR = "/mnt/dados/victor_baca/outputs/tsv/tsv"

# Mapeamento: subdiretório -> coluna a ser usada
SUBDIRS_CONFIG = {
    "ISEScan": "family",
    "Tns_tsv": "is_tn_name",
    "filtered_no_ISPa12_ISPpu17_tsv": "is_tn_name"
}

def normalizar_nome_cepa(nome_arquivo):
    """
    Normaliza o nome do arquivo para obter o nome base da cepa.
    
    Remove TODOS os sufixos e extensões comuns de forma iterativa.
    """
    nome = os.path.splitext(nome_arquivo)[0]  # Remove .tsv
    
    padroes = ['_blastn_filter', '_ISs', '_Tns', '_isescan', 
               '.fa', '.fasta', '.fna', '.tsv']
    
    # Aplica remoção até não haver mais mudanças
    mudou = True
    while mudou:
        mudou = False
        for padrao in padroes:
            if nome.endswith(padrao):
                nome = nome[:-len(padrao)]
                mudou = True
    
    return nome.strip()

def validar_elemento(elemento):
    """
    Valida se um elemento é válido (começa com IS ou Tn e tem conteúdo adicional).
    """
    if not elemento or not elemento.strip():
        return False
    
    elemento = elemento.strip()
    
    # Verifica se é apenas "IS" ou "Tn" sem identificador
    if elemento == "IS" or elemento == "Tn":
        return False
    
    # Deve começar com IS ou Tn seguido de algo mais
    if elemento.startswith("IS"):
        resto = elemento[2:].strip()
        if not resto:
            return False
        return True
    
    if elemento.startswith("Tn"):
        resto = elemento[2:].strip()
        if not resto:
            return False
        return True
    
    return False

def normalizar_nome_elemento(elemento, debug=False):
    """
    Normaliza o nome de um elemento IS/Tn para consolidar variantes.
    
    IMPORTANTE: Aplica múltiplas passadas para garantir remoção completa de variantes.
    """
    if not elemento:
        return ""
    
    nome_original = elemento.strip()
    nome = nome_original
    
    if debug:
        print(f"    DEBUG: Normalizando '{nome_original}'")
    
    # 1. Remove prefixo "ISFinder_"
    if nome.startswith("ISFinder_"):
        nome = nome[9:]
        if debug:
            print(f"    DEBUG: Após remover ISFinder_: '{nome}'")
    
    # 2. Remove número de acesso (tudo após o primeiro hífen)
    if '-' in nome:
        nome = nome.split('-')[0]
        if debug:
            print(f"    DEBUG: Após remover número de acesso: '{nome}'")
    
    # 3. Remove variantes - APLICAR MÚLTIPLAS VEZES até não haver mais mudanças
    mudou = True
    iteracoes = 0
    max_iteracoes = 10  # Previne loop infinito
    
    while mudou and iteracoes < max_iteracoes:
        mudou = False
        iteracoes += 1
        nome_antes = nome
        
        # Remove .var seguido de números (ex: .var1, .var2)
        nome_temp = re.sub(r'\.var\d+$', '', nome)
        if nome_temp != nome:
            nome = nome_temp
            mudou = True
            if debug:
                print(f"    DEBUG: Após remover .varN (iter {iteracoes}): '{nome}'")
        
        # Remove ponto seguido de números (ex: .1, .2, .10, .123)
        nome_temp = re.sub(r'\.\d+$', '', nome)
        if nome_temp != nome:
            nome = nome_temp
            mudou = True
            if debug:
                print(f"    DEBUG: Após remover .N (iter {iteracoes}): '{nome}'")
        
        # Remove _p no final
        nome_temp = re.sub(r'_p$', '', nome)
        if nome_temp != nome:
            nome = nome_temp
            mudou = True
            if debug:
                print(f"    DEBUG: Após remover _p (iter {iteracoes}): '{nome}'")
        
        # Remove underscore seguido de números
        nome_temp = re.sub(r'_\d+$', '', nome)
        if nome_temp != nome:
            nome = nome_temp
            mudou = True
            if debug:
                print(f"    DEBUG: Após remover _N (iter {iteracoes}): '{nome}'")
    
    # Remove espaços em branco extras
    nome = nome.strip()
    
    if debug and nome != nome_original:
        print(f"    DEBUG: RESULTADO: '{nome_original}' → '{nome}'")
    
    return nome

def processar_arquivo(caminho_arquivo, coluna_elemento, debug_normalizacao=False):
    """
    Processa um arquivo TSV e retorna contagem de elementos.
    """
    contagens = defaultdict(int)
    elementos_invalidos = set()
    mapeamento_normalizacao = {}  # Para debug: original -> normalizado
    
    try:
        with open(caminho_arquivo, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f, delimiter='\t')
            
            if coluna_elemento not in reader.fieldnames:
                print(f"  ⚠️  Coluna '{coluna_elemento}' não encontrada em {os.path.basename(caminho_arquivo)}")
                return contagens
            
            for linha in reader:
                elemento_original = linha[coluna_elemento].strip()
                
                if not elemento_original:
                    continue
                
                # Normaliza o nome do elemento
                elemento_normalizado = normalizar_nome_elemento(elemento_original, debug=debug_normalizacao)
                
                # Registra mapeamento
                if elemento_original != elemento_normalizado:
                    if elemento_normalizado not in mapeamento_normalizacao:
                        mapeamento_normalizacao[elemento_normalizado] = set()
                    mapeamento_normalizacao[elemento_normalizado].add(elemento_original)
                
                # Valida o elemento
                if not validar_elemento(elemento_normalizado):
                    elementos_invalidos.add(elemento_normalizado if elemento_normalizado else "(vazio)")
                    continue
                
                contagens[elemento_normalizado] += 1
        
        # Reporta elementos inválidos
        if elementos_invalidos:
            print(f"  ⚠️  Elementos inválidos ignorados: {', '.join(sorted(elementos_invalidos))}")
        
        # Reporta normalizações (mostra consolidações importantes)
        if mapeamento_normalizacao:
            consolidacoes_importantes = {k: v for k, v in mapeamento_normalizacao.items() if len(v) > 1}
            if consolidacoes_importantes:
                print(f"  📝 Consolidações realizadas:")
                for normalizado, originais in sorted(consolidacoes_importantes.items()):
                    print(f"     {normalizado} ← {', '.join(sorted(originais))}")
                    
    except Exception as e:
        print(f"  ❌ Erro ao processar {os.path.basename(caminho_arquivo)}: {e}")
    
    return contagens

def main():
    """Função principal."""
    print("=" * 80)
    print("Gerando is_counts.tsv a partir de múltiplos subdiretórios")
    print("Consolidando famílias (ISEScan) e elementos (Tns_tsv + filtered_*)")
    print("=" * 80)
    
    # Estrutura ÚNICA para armazenar todos os dados: {cepa: {elemento: contagem}}
    dados_gerais = defaultdict(lambda: defaultdict(int))
    elementos_todos = set()
    
    # Estatísticas separadas por fonte
    stats_por_fonte = {
        "ISEScan": {"arquivos": 0, "ocorrencias": 0, "unicos": set()},
        "Tns_tsv": {"arquivos": 0, "ocorrencias": 0, "unicos": set()},
        "filtered_no_ISPa12_ISPpu17_tsv": {"arquivos": 0, "ocorrencias": 0, "unicos": set()}
    }
    
    total_arquivos_processados = 0
    arquivos_por_cepa = defaultdict(list)
    
    # Processa cada subdiretório
    for subdir, coluna in SUBDIRS_CONFIG.items():
        caminho_subdir = os.path.join(BASE_DIR, subdir)
        
        tipo_descricao = "FAMÍLIAS DE TEs" if subdir == "ISEScan" else "ELEMENTOS IS/Tn"
        
        print(f"\n{'=' * 80}")
        print(f"📁 Processando: {subdir}/")
        print(f"   Coluna: '{coluna}'")
        print(f"   Tipo: {tipo_descricao}")
        print(f"{'=' * 80}")
        
        if not os.path.exists(caminho_subdir):
            print(f"   ⚠️  Diretório não encontrado: {caminho_subdir}")
            continue
        
        arquivos_tsv = [f for f in os.listdir(caminho_subdir) if f.endswith('.tsv')]
        
        if not arquivos_tsv:
            print(f"   ⚠️  Nenhum arquivo .tsv encontrado")
            continue
        
        print(f"   Encontrados {len(arquivos_tsv)} arquivos .tsv")
        
        arquivos_processados_subdir = 0
        
        # Processa cada arquivo
        for arquivo in sorted(arquivos_tsv):
            caminho_completo = os.path.join(caminho_subdir, arquivo)
            cepa = normalizar_nome_cepa(arquivo)
            
            arquivos_por_cepa[cepa].append(f"{subdir}/{arquivo}")
            
            # Debug desativado por padrão (mude para True se necessário)
            debug = False
            
            contagens = processar_arquivo(caminho_completo, coluna, debug_normalizacao=debug)
            
            if contagens:
                # Adiciona aos dados gerais (consolida tudo)
                for elemento, count in contagens.items():
                    dados_gerais[cepa][elemento] += count
                    elementos_todos.add(elemento)
                    stats_por_fonte[subdir]["unicos"].add(elemento)
                
                stats_por_fonte[subdir]["ocorrencias"] += sum(contagens.values())
                stats_por_fonte[subdir]["arquivos"] += 1
                arquivos_processados_subdir += 1
                total_arquivos_processados += 1
                
                print(f"   ✓ {arquivo} → {cepa}: {len(contagens)} itens únicos, {sum(contagens.values())} ocorrências")
            else:
                print(f"   - {arquivo} → {cepa}: sem dados válidos")
        
        print(f"\n   📊 Resumo {subdir}: {arquivos_processados_subdir} arquivos processados")
    
    # Verifica se há dados para processar
    if not dados_gerais:
        print("\n❌ Nenhum dado foi encontrado. Verifique os diretórios e arquivos.")
        return
    
    # Ordena elementos e cepas
    elementos_ordenados = sorted(elementos_todos)
    cepas_ordenadas = sorted(dados_gerais.keys())
    
    # Gera arquivo de saída ÚNICO
    arquivo_saida = "is_counts.tsv"
    print(f"\n{'=' * 80}")
    print(f"📝 Gerando arquivo de saída: {arquivo_saida}")
    print(f"{'=' * 80}")
    
    with open(arquivo_saida, 'w', newline='', encoding='utf-8') as f_out:
        writer = csv.writer(f_out, delimiter='\t')
        
        # Escreve cabeçalho
        writer.writerow(["genome"] + elementos_ordenados)
        
        # Escreve dados de cada cepa
        for cepa in cepas_ordenadas:
            linha = [cepa]
            for elemento in elementos_ordenados:
                linha.append(dados_gerais[cepa].get(elemento, 0))
            writer.writerow(linha)
    
    # Estatísticas finais
    print("\n" + "=" * 80)
    print("✅ ARQUIVO GERADO COM SUCESSO!")
    print("=" * 80)
    print(f"Arquivo: {arquivo_saida}")
    print(f"Total de genomas únicos: {len(cepas_ordenadas)}")
    print(f"Total de elementos/famílias únicos: {len(elementos_ordenados)}")
    print(f"Total de arquivos TSV processados: {total_arquivos_processados}")
    
    # Estatísticas por fonte
    print(f"\n📊 Estatísticas por fonte:")
    for fonte, stats in stats_por_fonte.items():
        if stats["arquivos"] > 0:
            tipo = "famílias" if fonte == "ISEScan" else "elementos"
            print(f"   {fonte}:")
            print(f"     - Arquivos: {stats['arquivos']}")
            print(f"     - {tipo.capitalize()} únicos: {len(stats['unicos'])}")
            print(f"     - Ocorrências totais: {stats['ocorrencias']}")
    
    # Mostra cepas com múltiplos arquivos (primeiros 10)
    cepas_multiplos = {cepa: arqs for cepa, arqs in arquivos_por_cepa.items() if len(arqs) > 1}
    if cepas_multiplos:
        print(f"\n📋 Cepas consolidadas de múltiplos arquivos: {len(cepas_multiplos)}")
        print("   (Mostrando primeiros 10:)")
        for i, (cepa, arquivos) in enumerate(sorted(cepas_multiplos.items())[:10], 1):
            print(f"   {i}. {cepa} ({len(arquivos)} arquivos):")
            for arq in sorted(arquivos):
                print(f"      - {arq}")
    
    # Top 10 elementos/famílias mais frequentes
    print("\n🔝 Top 10 elementos/famílias mais frequentes:")
    contagem_total_elementos = defaultdict(int)
    for cepa_dados in dados_gerais.values():
        for elemento, count in cepa_dados.items():
            contagem_total_elementos[elemento] += count
    
    top_elementos = sorted(contagem_total_elementos.items(), key=lambda x: x[1], reverse=True)[:10]
    for i, (elemento, count) in enumerate(top_elementos, 1):
        print(f"   {i:2d}. {elemento}: {count} ocorrências")
    
    # Lista todos os elementos únicos encontrados
    print(f"\n📋 Todos os {len(elementos_ordenados)} elementos/famílias IS/Tn únicos:")
    
    # Separa por prefixo para melhor visualização
    elementos_is = [e for e in elementos_ordenados if e.startswith('IS')]
    elementos_tn = [e for e in elementos_ordenados if e.startswith('Tn')]
    
    if elementos_is:
        print(f"\n   Elementos/Famílias IS ({len(elementos_is)}):")
        for i in range(0, len(elementos_is), 10):
            print(f"   {', '.join(elementos_is[i:i+10])}")
    
    if elementos_tn:
        print(f"\n   Elementos Tn ({len(elementos_tn)}):")
        for i in range(0, len(elementos_tn), 10):
            print(f"   {', '.join(elementos_tn[i:i+10])}")
    
    print("\n" + "=" * 80)

if __name__ == "__main__":
    main()
#####################################################################################################################################################

### CTRL + O = Salvar arquivo
### CTRL + X = Sair do nano 

chmod +x generate_is_counts.py
python3 generate_is_counts.py

# Por ultimo vamos criar o script para adicionar as familias de Tns ao arquivo is_counts.tsv
nano add_tn_families.py

#####################################################################################################################################################
#!/usr/bin/env python3
"""
Script para adicionar famílias de elementos Tn ao arquivo is_counts.tsv

O script:
1. Lê o arquivo is_counts.tsv
2. Cria colunas agregadas por família de Tn
3. Soma os valores de elementos que pertencem à mesma família
4. Gera arquivo atualizado com as colunas de família
"""

import csv
import sys
from collections import defaultdict

# Mapeamento: Elemento Tn -> Família
TN_FAMILIES = {
    "Tn1546": "Tn3",
    "Tn2": "Tn3",
    "Tn2670": "Compound_Transposon",
    "Tn4001": "Compound_Transposon",
    "Tn501": "Tn3",
    "Tn5041": "Tn3",
    "Tn5393": "Tn3",
    "Tn551": "Tn3",
    "Tn5531": "Compound_Transposon",
    "Tn6214": "Tn402",
    "Tn6246": "Compound_Transposon",
    "Tn6674": "Tn554",
    "Tn7114": "Tn3",
    "TnSs1": "Compound_Transposon",
    "TnSsu5": "Compound_Transposon"
}

def processar_is_counts(arquivo_entrada, arquivo_saida):
    """
    Processa o arquivo is_counts.tsv e adiciona colunas de famílias Tn.
    """
    print("=" * 80)
    print("Adicionando famílias de Tn ao is_counts.tsv")
    print("=" * 80)
    
    # Lê o arquivo original
    print(f"\n📖 Lendo arquivo: {arquivo_entrada}")
    
    try:
        with open(arquivo_entrada, 'r', encoding='utf-8') as f:
            reader = csv.reader(f, delimiter='\t')
            
            # Lê cabeçalho
            header = next(reader)
            
            # Lê todas as linhas de dados
            dados = list(reader)
            
        print(f"   ✓ {len(dados)} genomas carregados")
        print(f"   ✓ {len(header)-1} elementos/famílias encontrados no arquivo")
        
    except FileNotFoundError:
        print(f"❌ Erro: Arquivo '{arquivo_entrada}' não encontrado!")
        sys.exit(1)
    except Exception as e:
        print(f"❌ Erro ao ler arquivo: {e}")
        sys.exit(1)
    
    # Identifica quais elementos Tn estão presentes no arquivo
    elementos_tn_presentes = []
    indices_tn = {}  # elemento -> índice na coluna
    
    for i, col in enumerate(header[1:], start=1):  # Pula coluna 'genome'
        if col in TN_FAMILIES:
            elementos_tn_presentes.append(col)
            indices_tn[col] = i
    
    print(f"\n📋 Elementos Tn encontrados no arquivo: {len(elementos_tn_presentes)}")
    for elemento in sorted(elementos_tn_presentes):
        familia = TN_FAMILIES[elemento]
        print(f"   - {elemento} → {familia}")
    
    # Identifica famílias únicas
    familias_unicas = sorted(set(TN_FAMILIES[tn] for tn in elementos_tn_presentes))
    
    print(f"\n🧬 Famílias de Tn a serem adicionadas: {len(familias_unicas)}")
    for familia in familias_unicas:
        elementos = [tn for tn in elementos_tn_presentes if TN_FAMILIES[tn] == familia]
        print(f"   - {familia}: {', '.join(sorted(elementos))}")
    
    # Cria novo cabeçalho com colunas de família
    novo_header = header.copy()
    
    # Adiciona colunas de família (com sufixo _family para distinguir)
    for familia in familias_unicas:
        novo_header.append(f"{familia}")
    
    # Processa dados e calcula valores das famílias
    novos_dados = []
    
    print(f"\n⚙️  Calculando valores das famílias para cada genoma...")
    
    for linha in dados:
        nova_linha = linha.copy()
        
        # Para cada família, soma os valores dos elementos que pertencem a ela
        for familia in familias_unicas:
            # Encontra todos os elementos desta família
            elementos_da_familia = [tn for tn in elementos_tn_presentes if TN_FAMILIES[tn] == familia]
            
            # Soma os valores
            soma_familia = 0
            for elemento in elementos_da_familia:
                idx = indices_tn[elemento]
                try:
                    valor = int(linha[idx])
                    soma_familia += valor
                except (ValueError, IndexError):
                    pass
            
            nova_linha.append(soma_familia)
        
        novos_dados.append(nova_linha)
    
    # Escreve arquivo de saída
    print(f"\n💾 Salvando arquivo: {arquivo_saida}")
    
    try:
        with open(arquivo_saida, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f, delimiter='\t')
            writer.writerow(novo_header)
            writer.writerows(novos_dados)
        
        print(f"   ✓ Arquivo salvo com sucesso!")
        
    except Exception as e:
        print(f"❌ Erro ao salvar arquivo: {e}")
        sys.exit(1)
    
    # Estatísticas finais
    print("\n" + "=" * 80)
    print("✅ PROCESSO CONCLUÍDO!")
    print("=" * 80)
    print(f"Arquivo original: {arquivo_entrada}")
    print(f"   - Colunas: {len(header)}")
    print(f"   - Linhas: {len(dados)}")
    print(f"\nArquivo atualizado: {arquivo_saida}")
    print(f"   - Colunas: {len(novo_header)} (+{len(familias_unicas)} colunas de família)")
    print(f"   - Linhas: {len(novos_dados)}")
    
    print(f"\n📊 Colunas de família adicionadas:")
    for familia in familias_unicas:
        print(f"   - {familia}")
    
    # Mostra exemplo de dados (primeira linha)
    if novos_dados:
        print(f"\n📝 Exemplo (primeiro genoma: {novos_dados[0][0]}):")
        for familia in familias_unicas:
            col_idx = novo_header.index(f"{familia}")
            valor = novos_dados[0][col_idx]
            if valor > 0:
                elementos = [tn for tn in elementos_tn_presentes if TN_FAMILIES[tn] == familia]
                print(f"   {familia} = {valor} (soma de: {', '.join(elementos)})")
    
    print("=" * 80)

def main():
    """Função principal."""
    # Configuração de arquivos
    arquivo_entrada = "is_counts.tsv"
    arquivo_saida = "is_counts_with_familiesTns.tsv"
    
    # Verifica se há argumentos na linha de comando
    if len(sys.argv) > 1:
        arquivo_entrada = sys.argv[1]
    if len(sys.argv) > 2:
        arquivo_saida = sys.argv[2]
    
    print(f"Arquivo de entrada: {arquivo_entrada}")
    print(f"Arquivo de saída: {arquivo_saida}")
    
    # Processa
    processar_is_counts(arquivo_entrada, arquivo_saida)

if __name__ == "__main__":
    main()
#####################################################################################################################################################

chmod +x add_tn_families.py
./add_tn_families.py


In [ ]:
# Criar script para converter is_counts_with_familiesTns.tsv em formato iTOL com visualização quantitativa 
nano convert_to_itol.py

#####################################################################################################################################################

#!/usr/bin/env python3
"""
Script para converter is_counts_with_familiesTns.tsv em formato iTOL com visualização de 3 estados.

Gera arquivo itol_is_binary.txt compatível com https://itol.embl.de

Funcionalidades:
- 0 TEs: em branco (nenhum símbolo)
- 1 TE:  somente Margem/Borda do símbolo
- >1 TEs: símbolo totalmente pintado
- Cores distintas para famílias vs elementos
- Ordenação: famílias primeiro, depois elementos

Utiliza DATASET_BINARY do iTOL:
    -1 = em branco (0 cópias)
     0 = somente borda/margem (1 cópia)
     1 = totalmente preenchido (>1 cópias)
"""

import csv
import sys

# Configurações de entrada/saída
INPUT_FILE = "/mnt/dados/victor_baca/outputs/tsv/tsv/is_counts_with_familiesTns.tsv"
OUTPUT_FILE = "/mnt/dados/victor_baca/outputs/tsv/tsv/itol_is_binary.txt"

# Definição de famílias de TEs (serão destacadas)
FAMILIAS_TES = {
    # Famílias Tn
    "Tn3", "Tn3_family", "Compound_Transposon", "Compound_Transposon_family",
    "Tn402", "Tn402_family", "Tn554", "Tn554_family",

    # Famílias IS
    "IS3", "IS5", "IS6", "IS21", "IS30", "IS66", "IS110", "IS256", "IS630", "IS982", "IS1182", "IS1380", "IS1595", "IS200/IS605", "ISL3", "ISAS1", "ISAs1"
}

# Cores para FAMÍLIAS (cores vibrantes e destacadas)
CORES_FAMILIAS = [
    "#e6194b",  # Vermelho intenso
    "#3cb44b",  # Verde intenso
    "#ffe119",  # Amarelo intenso
    "#4363d8",  # Azul intenso
    "#f58231",  # Laranja intenso
    "#911eb4",  # Roxo intenso
    "#46f0f0",  # Ciano intenso
    "#f032e6",  # Magenta intenso
    "#bcf60c",  # Lima intenso
    "#fabebe",  # Rosa intenso
    "#008080",  # Teal
    "#e6beff",  # Lavanda
    "#9a6324",  # Marrom
    "#800000",  # Marrom escuro
    "#aaffc3",  # Menta
]

# Cores para ELEMENTOS (cores mais suaves/pastéis)
CORES_ELEMENTOS = [
    "#ffd8b1",  # Pêssego
    "#fffac8",  # Creme
    "#aaffc3",  # Menta clara
    "#dcbeff",  # Lilás claro
    "#ffb3ba",  # Rosa claro
    "#bae1ff",  # Azul claro
    "#c9c9c9",  # Cinza claro
    "#ffffba",  # Amarelo claro
    "#baffc9",  # Verde claro
    "#ffdfba",  # Laranja claro
    "#e0bbff",  # Roxo claro
    "#d4f1f4",  # Azul gelo
    "#ffc9de",  # Rosa bebê
    "#c7ceea",  # Azul acinzentado
    "#b5ead7",  # Verde água claro
]

def eh_familia(coluna):
    """Verifica se a coluna representa uma família de TE."""
    return coluna in FAMILIAS_TES

def converter_para_binario(valor_str):
    """
    Converte valor numérico para código de 3 estados do iTOL DATASET_BINARY:

        0 cópias → -1  (em branco, nenhum símbolo)
        1 cópia  →  0  (somente borda/margem do símbolo)
       >1 cópias →  1  (símbolo totalmente preenchido)
    """
    try:
        num = int(valor_str)
        if num == 0:
            return "-1"   # em branco
        elif num == 1:
            return "0"    # somente borda
        else:
            return "1"    # totalmente preenchido
    except ValueError:
        return "-1"       # valor inválido → em branco

def processar_arquivo():
    """Função principal de processamento."""
    print("=" * 80)
    print("Convertendo is_counts_with_familiesTns.tsv para formato iTOL (DATASET_BINARY)")
    print("Visualização de 3 estados:")
    print("   0 cópias  → em branco (nenhum símbolo)")
    print("   1 cópia   → somente Borda/Margem do símbolo")
    print("  >1 cópias  → símbolo totalmente pintado")
    print("=" * 80)

    # Lê arquivo de entrada
    print(f"\n📖 Lendo arquivo: {INPUT_FILE}")

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            reader = csv.reader(f, delimiter='\t')
            header = next(reader)
            dados = list(reader)

        print(f"   ✓ {len(dados)} genomas carregados")
        print(f"   ✓ {len(header)-1} colunas de IS/Tn encontradas")

    except FileNotFoundError:
        print(f"❌ Erro: Arquivo '{INPUT_FILE}' não encontrado!")
        sys.exit(1)
    except Exception as e:
        print(f"❌ Erro ao ler arquivo: {e}")
        sys.exit(1)

    # Separa famílias e elementos
    colunas_info = []
    for i, col in enumerate(header[1:], start=1):
        tipo = "FAMÍLIA" if eh_familia(col) else "ELEMENTO"
        colunas_info.append({
            'indice': i,
            'nome': col,
            'tipo': tipo
        })

    familias = [c for c in colunas_info if c['tipo'] == "FAMÍLIA"]
    elementos = [c for c in colunas_info if c['tipo'] == "ELEMENTO"]

    print(f"\n📊 Classificação:")
    print(f"   - Famílias de TEs: {len(familias)} (cores vibrantes)")
    print(f"   - Elementos de TEs: {len(elementos)} (cores suaves)")

    if familias:
        print(f"\n🧬 Famílias identificadas:")
        for fam in sorted([f['nome'] for f in familias]):
            print(f"   ⭐ {fam}")

    # Reordena colunas: famílias primeiro, depois elementos
    colunas_ordenadas = familias + elementos
    nomes_ordenados  = [c['nome'] for c in colunas_ordenadas]
    indices_ordenados = [c['indice'] for c in colunas_ordenadas]

    # Gera cores (mantém lógica original)
    cores = []
    idx_fam = 0
    idx_ele = 0
    for col_info in colunas_ordenadas:
        if col_info['tipo'] == "FAMÍLIA":
            cores.append(CORES_FAMILIAS[idx_fam % len(CORES_FAMILIAS)])
            idx_fam += 1
        else:
            cores.append(CORES_ELEMENTOS[idx_ele % len(CORES_ELEMENTOS)])
            idx_ele += 1

    # Define formas dos símbolos por tipo
    # Opções iTOL: 1=círculo, 2=quadrado, 3=estrela, 4=triâng.dir., 5=triâng.esq., 6=check
    # Famílias → estrela (3) | Elementos → círculo (1)
    formas = ["3" if c['tipo'] == "FAMÍLIA" else "1" for c in colunas_ordenadas]

    # Converte dados para o sistema de 3 estados
    print(f"\n⚙️  Convertendo valores para 3 estados (−1 / 0 / 1)...")

    dados_convertidos = []
    contagem = {"-1": 0, "0": 0, "1": 0}
    for linha in dados:
        nova_linha = [linha[0]]  # Nome do genoma
        for idx in indices_ordenados:
            valor_bin = converter_para_binario(linha[idx])
            nova_linha.append(valor_bin)
            contagem[valor_bin] += 1
        dados_convertidos.append(nova_linha)

    total_celulas = sum(contagem.values())
    print(f"   ✓ Células em branco      (0 cópias)  [-1]: {contagem['-1']:>6}  ({100*contagem['-1']/total_celulas:.1f}%)")
    print(f"   ✓ Somente borda          (1 cópia)   [ 0]: {contagem['0']:>6}  ({100*contagem['0']/total_celulas:.1f}%)")
    print(f"   ✓ Totalmente preenchido (>1 cópias)  [ 1]: {contagem['1']:>6}  ({100*contagem['1']/total_celulas:.1f}%)")

    # Escreve arquivo iTOL
    print(f"\n💾 Gerando arquivo iTOL: {OUTPUT_FILE}")

    try:
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as out:

            # ── Cabeçalho DATASET_BINARY ─────────────────────────────────────
            out.write("DATASET_BINARY\n")
            out.write("SEPARATOR TAB\n")
            out.write("\n")

            out.write("DATASET_LABEL\tIS/Tn elements and families\n")
            out.write("\n")

            out.write("COLOR\t#ff0000\n")
            out.write("\n")

            # ── Configuração dos campos ──────────────────────────────────────
            out.write("# Formas dos símbolos: famílias=estrela (3), elementos=círculo (1)\n")
            out.write("FIELD_SHAPES\t" + "\t".join(formas) + "\n")
            out.write("\n")

            out.write("# Labels das colunas (famílias primeiro, depois elementos)\n")
            out.write("FIELD_LABELS\t" + "\t".join(nomes_ordenados) + "\n")
            out.write("\n")

            out.write("# Cores: vibrantes para famílias, suaves para elementos\n")
            out.write("FIELD_COLORS\t" + "\t".join(cores) + "\n")
            out.write("\n")

            # ── Dados ────────────────────────────────────────────────────────
            out.write("# Dados:\n")
            out.write("#   -1 = em branco       (0 cópias do TE)\n")
            out.write("#    0 = somente borda   (1 cópia do TE)\n")
            out.write("#    1 = totalmente pint. (>1 cópias do TE)\n")
            out.write("DATA\n")

            for linha in dados_convertidos:
                out.write("\t".join(linha) + "\n")

        print(f"   ✓ Arquivo salvo com sucesso!")

    except Exception as e:
        print(f"❌ Erro ao salvar arquivo: {e}")
        sys.exit(1)

    # Estatísticas finais
    print("\n" + "=" * 80)
    print("✅ CONVERSÃO CONCLUÍDA!")
    print("=" * 80)
    print(f"Arquivo gerado: {OUTPUT_FILE}")
    print(f"Total de genomas : {len(dados_convertidos)}")
    print(f"Total de campos  : {len(nomes_ordenados)}")
    print(f"   - Famílias    : {len(familias)}")
    print(f"   - Elementos   : {len(elementos)}")

    print(f"\n📊 Legenda de Visualização:")
    print(f"   ⬜ Nenhum símbolo  → 0 cópias do TE")
    print(f"   ○  Somente borda   → 1 cópia do TE")
    print(f"   ●  Totalmente pint.→ >1 cópias do TE")

    print(f"\n📋 Próximos passos:")
    print(f"   1. Acesse: https://itol.embl.de")
    print(f"   2. Faça upload da árvore: core_gene_alignment.aln.treefile")
    print(f"   3. Depois faça upload do arquivo: {OUTPUT_FILE}")
    print(f"   ⭐ Famílias → estrela")
    print(f"   ● Elementos → círculo")

    print("\n" + "=" * 80)

def main():
    """Execução principal."""
    global INPUT_FILE, OUTPUT_FILE

    if len(sys.argv) > 1:
        INPUT_FILE = sys.argv[1]
    if len(sys.argv) > 2:
        OUTPUT_FILE = sys.argv[2]

    processar_arquivo()

if __name__ == "__main__":
    main()

#####################################################################################################################################################

chmod +x convert_to_itol.py
python3 convert_to_itol.py


In [ ]:
#### 5. Visualizar no iTOL
## Va para https://itol.embl.de
## Clique em “Upload Tree”
## Suba o arquivo: core_gene_alignment.aln.treefile
## Clique em “Datasets” > “Upload” > selecione o .txt com ISs do arquivo .tsv (itol_is_binary.txt)
## Visualize com circulos coloridos ou heatmap por IS

In [ ]:
# Copiar um arquivo do servidor para maquina local (windowns)
scp victorbaca@10.20.34.31:/mnt/dados/victor_baca/outputs/tsv/tsv/itol_is_binary.txt "C:\Users\vitor\Downloads"

scp victorbaca@10.20.34.31:/mnt/dados/victor_baca/outputs/gff_collection/roary_output/core_gene_alignment.aln.treefile "C:\Users\vitor\Downloads"

scp victorbaca@10.20.34.31:/mnt/dados/victor_baca/outputs/tsv/tsv/is_counts_with_familiesTns.tsv "C:\Users\vitor\Downloads"
